In [3]:
import pandas as pd

# 1. LOAD AND CLEAN ADVANCED TEAM SEASON STATS
df = pd.read_csv('data/teams_advanced.csv')

df['team'] = (
    df['team']
    .astype(str)
    .str.replace(' ', ' ', regex=False)
    .str.replace('*', '', regex=False)
    .str.strip()
)

df['team'] = df['team'].replace({
    'Albany (NY)': 'Albany'
})

df = df[df['team'] != 'School'].copy()

num_cols = df.columns.drop('team')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 2. LOAD TOURNAMENT GAMES
games = pd.read_csv('data/tournament_games.csv')

games['winner'] = games['winner'].replace({'Albany (NY)': 'Albany'})
games['loser'] = games['loser'].replace({'Albany (NY)': 'Albany'})

# 3. MERGE TEAM STATS (WINNER + LOSER)
games_w = games.merge(
    df,
    left_on=['winner', 'season'],
    right_on=['team', 'year'],
    how='left'
)

games_wl = games_w.merge(
    df,
    left_on=['loser', 'season'],
    right_on=['team', 'year'],
    how='left',
    suffixes=('_winner', '_loser')
)

# 4. CREATE DIFFERENCE FEATURES
features = [
    'SRS', 'win_pct', 'eFG%', 'TS%', 'TRB%', 'AST%', 'TOV%', 'ORtg'
]

for col in features:
    games_wl[f'{col}_diff'] = games_wl[f'{col}_winner'] - games_wl[f'{col}_loser']

games_wl['seed_diff'] = games_wl['winner_seed'] - games_wl['loser_seed']

# 5. CREATE SYMMETRIC DATASET
games_wl['win_label'] = 1
loser_view = games_wl.copy()
for col in features:
    loser_view[f'{col}_diff'] *= -1
loser_view['seed_diff'] *= -1
loser_view['win_label'] = 0
model_df = pd.concat([games_wl, loser_view], ignore_index=True)

# 6. BUILD FINAL ML DATASET
feature_cols = sorted(set([c for c in model_df.columns if c.endswith('_diff')]))
final_df = model_df[feature_cols + ['win_label', 'season']]
final_df = final_df.dropna()

# 7. SAVE TO CSV
final_df.to_csv('data/tournament_model_adv_ml.csv', index=False)
